**Week 11**<br />Monday, Nov. 9

# Language Modeling

LX 394/694<br />
Coding for Linguists

Sophie Hao


## Announcements

- HW 2 is **due** (for real this time)!
- HW 3 released either today or tomorrow
- Due Monday, Nov. 23, at 9 AM
- Train and test a spam email classifier

## Recap: Language Modeling

**Language modeling** is an NLP task where, given a sequence of tokens $x = x_1x_2 \dots x_l$ where $x_1 = \texttt{'<s>'}$, the **language model** must predict the next token, $y$.

Given input $x$, the language model's output is:

$$
\hat{y} = \begin{Bmatrix}
   \text{the:} & \mathbb{P}[\text{the} | x] \\
   \text{of:} & \mathbb{P}[\text{of} | x] \\
   \text{in:} & \mathbb{P}[\text{in} | x] \\
   \vdots & \vdots
\end{Bmatrix}
$$


## Recap: Why Language Modeling?

We will learn about the following applications of language modeling:
- text generation
- classification (e.g., spam filtering)
- noisy channel model (e.g., spelling correction)

## Recap: Text Generation With a Language Model

You can generate text with a language model by repeatedly predicting next words.

Two common ways of generating text are...
- **Greedy generation:** always choose the token with the highest probability
- **Non-greedy generation:** choose a token based on the probability distribution from the language model

## N-Gram Language Models

Like unigram and bigram POS taggers, there are also unigram and bigram language models.

**Unigram Language Model:**
$$\mathbb{P}[w | x] = \mathbb{P}[w]$$

**Bigram Language Model:**
$$\mathbb{P}[w | x] = \mathbb{P}[w | x_l]$$

**Trigram Language Model:**
$$\mathbb{P}[w | x] = \mathbb{P}[w | x_{l - 1}x_l]$$

## Training an N-Gram Language Model

Let $c$ be a training corpus.

**Unigram Model:** ($n = 1$)

$$\mathbb{P}[w] = \frac{c\texttt{.count}(w)}{\texttt{len}(c)}$$

**Bigram Model:** ($n = 2$)

$$\mathbb{P}[w|x_l] = \frac{c\texttt{.count}(x_l w)}{c\texttt{.count}(x_l)}$$

**Trigram Model:** ($n = 3$)

$$\mathbb{P}[w|x_{l - 1}x_l] = \frac{c\texttt{.count}(x_{l - 1} x_l w)}{c\texttt{.count}(x_{l - 1}x_l)}$$

**N-Gram Model:** ($n > 1$)

$$\mathbb{P}[w|x_{l - n + 1} x_{l - n + 2} \dots x_l] = \frac{c\texttt{.count}(x_{l - n + 1} x_{l - n + 2} \dots x_l w)}{c\texttt{.count}(x_{l - n + 1} x_{l - n + 2} \dots x_l)}$$

## Language Modeling in NLTK

In [1]:
from nltk.lm.preprocessing import padded_everygram_pipeline
from nltk.lm import MLE


def train_lm(n: int, corpus) -> MLE:
    """ Train an n-gram model on corpus. """
    training_data, vocab = padded_everygram_pipeline(n, corpus)
    lm = MLE(n)
    lm.fit(training_data, vocab)
    return lm

In [3]:
# What does padded_everygram_pipeline do?
toy_corpus = ["Hello world !".split(), "How are you ?".split()]
data, _ = padded_everygram_pipeline(3, toy_corpus)

# Data consists of a list of n-grams
for i, sentence in enumerate(data):
    print(f"Sentence {i}: ", list(sentence))

Sentence 0:  [('<s>',), ('<s>', '<s>'), ('<s>', '<s>', 'Hello'), ('<s>',), ('<s>', 'Hello'), ('<s>', 'Hello', 'world'), ('Hello',), ('Hello', 'world'), ('Hello', 'world', '!'), ('world',), ('world', '!'), ('world', '!', '</s>'), ('!',), ('!', '</s>'), ('!', '</s>', '</s>'), ('</s>',), ('</s>', '</s>'), ('</s>',)]
Sentence 1:  [('<s>',), ('<s>', '<s>'), ('<s>', '<s>', 'How'), ('<s>',), ('<s>', 'How'), ('<s>', 'How', 'are'), ('How',), ('How', 'are'), ('How', 'are', 'you'), ('are',), ('are', 'you'), ('are', 'you', '?'), ('you',), ('you', '?'), ('you', '?', '</s>'), ('?',), ('?', '</s>'), ('?', '</s>', '</s>'), ('</s>',), ('</s>', '</s>'), ('</s>',)]


## Train a Trigram Model on the Brown Corpus

In [4]:
from nltk.corpus import brown

brown_lm = train_lm(3, brown.sents())

In [11]:
# Generate text
brown_lm.generate(10)

['not', 'mean', 'to', 'live', 'in', 'a', 'timbered', 'pit', 'on', 'the']

## Generating with a Prompt

In [25]:
import nltk 

prompt_raw = "Boston University is located in the Commonwealth of"
prompt_tokens = nltk.word_tokenize(prompt_raw)
brown_lm.generate(1, text_seed=prompt_tokens)

'Massachusetts'

In [16]:
# What is P('Massachusetts' | ('Commonwealth', 'of'))?
brown_lm.score("Massachusetts", context=("Commonwealth", "of"))

0.25

## Exercise

This is the code from the previous slide.

In [26]:
# What is P('Massachusetts' | ('state', 'of'))?
brown_lm.score("Massachusetts", context=("state", "of"))

0.012195121951219513

Using our `brown_lm` trigram model, how do we estimate the probability that the first token of a sentence is `'The'`?

In [27]:
# Solution: it's P('The' | ('<s>', '<s>'))
brown_lm.score("The", context=("<s>", "<s>"))

0.11412626438786187

## Exercise: Probability of a Text

How do we calculate the probability that our `brown_lm` trigram model will generate:

```python
"Boston University is located in the state of Massachusetts."
``` 

non-greedily?

## Next Time

Language Modeling, Continued